# Script 03: Create IMC AnnData files from segmentation CSVs

This script generates per-ROI AnnData (.h5ad) files from the Mesmer/DeepCell segmentation outputs
(cell_info, protein_expression, polygon_coords CSVs).

**Filtering**: Only well-segmented ROIs (>100 cells) are included. Underscore-variant filenames
(e.g. `ROI3_Control_2`) are excluded as they represent failed/partial segmentation runs.

**Output**: One .h5ad per ROI in `script01_imc_adatas/`

In [ ]:
import anndata as ad
import pandas as pd
import numpy as np
import os
import re
from scipy.sparse import csr_matrix

In [ ]:
root = 'T:/Analysis/116_MDACC_Chiba/chiba/out_feb2025__protein_analysis_imc_orig/'

cell_info_dir = os.path.join(root, 'script01_imc_cell_info')
protein_expr_dir = os.path.join(root, 'script01_imc_protein_expression')
polygon_dir = os.path.join(root, 'script01_imc_polygon_img_coords')

adata_out_dir = os.path.join(root, 'script01_imc_adatas')
os.makedirs(adata_out_dir, exist_ok=True)

print(f'Cell info dir: {cell_info_dir}')
print(f'Protein expr dir: {protein_expr_dir}')
print(f'Output dir: {adata_out_dir}')

## Discover and filter ROIs

We identify well-segmented ROIs (>100 cells) and exclude underscore-variant duplicates.

In [ ]:
# List all cell_info CSVs and count cells per file
cell_info_files = sorted([f for f in os.listdir(cell_info_dir) if f.endswith('.csv')])

roi_info = []
for f in cell_info_files:
    df = pd.read_csv(os.path.join(cell_info_dir, f))
    # Extract sample name: remove '_cell_info.csv' suffix
    sample_name = f.replace('_cell_info.csv', '')
    roi_info.append({'filename': f, 'sample_name': sample_name, 'n_cells': len(df)})

roi_df = pd.DataFrame(roi_info)
print(f'Total CSVs found: {len(roi_df)}')
roi_df.sort_values('n_cells', ascending=False)

In [ ]:
# Identify underscore-variant duplicates
# These are ROIs where a space in the original name was replaced with an underscore
# (e.g. 'ROI3_Control_2' with 1 cell vs 'ROI3_Control 2' with 2138 cells)
# Strategy: for each ROI prefix (e.g. ROI3, ROI16, ROI20, ROI24, ROI36),
# if there are two versions, keep the one with more cells.

# Known underscore variants to exclude (confirmed failed/partial runs with very few cells)
underscore_variants = [
    'ROI3_Control_2',
    'ROI16_Oligo_4-Edge',
    'ROI20_Oligo_3-Core',
    'ROI24_Oligo_3-Edge',
    'ROI36_Oligo_1-Core',
]

# Filter: exclude underscore variants AND require >100 cells
MIN_CELLS = 100
roi_filtered = roi_df[
    (~roi_df['sample_name'].isin(underscore_variants)) & 
    (roi_df['n_cells'] >= MIN_CELLS)
].copy()

roi_filtered = roi_filtered.sort_values('n_cells', ascending=False).reset_index(drop=True)
print(f'Well-segmented ROIs: {len(roi_filtered)} ({roi_filtered["n_cells"].sum()} total cells)')
roi_filtered[['sample_name', 'n_cells']]

## Define marker metadata

25 protein markers from the IMC panel. Channel indices refer to the original OME-TIFF channels.

In [ ]:
# Marker names in order (matches column order in protein_expression CSVs after cell_id)
markers = [
    'DNA', 'VIMENTIN', 'EGFR', 'GFAP', 'NEUN', 'CD163', 'MAP2',
    'O4', 'CD45', 'CD4', 'E-CAD', 'CD68', 'CD8A', 'CX3CR1',
    'CCK', 'ARGINASE1', 'CD34', 'KI67', 'IBA1', 'CD3', 'H3',
    'P2Y12', 'ICSK1', 'ICSK2', 'ICSK3'
]

# Original channel indices in the OME-TIFF (from script01)
channel_indices = [
    30, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21,
    22, 23, 24, 25, 26, 27, 28, 29, 32, 33, 34
]

# Marker categories for downstream analysis
marker_type = {
    'DNA': 'nuclear',
    'VIMENTIN': 'structural', 'EGFR': 'receptor', 'GFAP': 'glial',
    'NEUN': 'neuronal', 'CD163': 'immune', 'MAP2': 'neuronal',
    'O4': 'oligo', 'CD45': 'immune', 'CD4': 'immune',
    'E-CAD': 'epithelial', 'CD68': 'immune', 'CD8A': 'immune',
    'CX3CR1': 'immune', 'CCK': 'neuropeptide', 'ARGINASE1': 'immune',
    'CD34': 'endothelial', 'KI67': 'proliferation', 'IBA1': 'microglia',
    'CD3': 'immune', 'H3': 'nuclear', 'P2Y12': 'microglia',
    'ICSK1': 'isotype_control', 'ICSK2': 'isotype_control', 'ICSK3': 'isotype_control'
}

print(f'{len(markers)} markers defined')

## Build AnnData objects per ROI

In [ ]:
cofactor = 5  # arcsinh cofactor (standard for CyTOF/IMC)

for idx, row in roi_filtered.iterrows():
    sample_name = row['sample_name']
    
    # --- Load CSVs ---
    cell_info = pd.read_csv(os.path.join(cell_info_dir, f'{sample_name}_cell_info.csv'))
    protein_expr = pd.read_csv(os.path.join(protein_expr_dir, f'{sample_name}_protein_expression.csv'))
    
    polygon_file = os.path.join(polygon_dir, f'{sample_name}_polygons_img_coord.csv')
    if os.path.exists(polygon_file):
        polygons = pd.read_csv(polygon_file)
    else:
        polygons = None
    
    # --- Merge and validate ---
    # Ensure cell_info and protein_expr have same cells in same order
    assert set(cell_info['cell_id']) == set(protein_expr['cell_id']), \
        f'{sample_name}: cell_id mismatch between cell_info and protein_expression'
    
    # Sort both by cell_id
    cell_info = cell_info.sort_values('cell_id').reset_index(drop=True)
    protein_expr = protein_expr.sort_values('cell_id').reset_index(drop=True)
    
    n_cells = len(cell_info)
    
    # --- Build expression matrix (.X) ---
    # Columns in protein_expr: cell_id, then marker names
    expr_cols = [c for c in protein_expr.columns if c != 'cell_id']
    X = protein_expr[expr_cols].values.astype(np.float64)
    
    # Verify marker order matches our expected list
    assert expr_cols == markers, \
        f'{sample_name}: marker column mismatch. Got {expr_cols}'
    
    # --- Build .obs ---
    obs = pd.DataFrame({
        'cell_id': cell_info['cell_id'].values,
        'centroid_x': cell_info['centroid_x'].values,
        'centroid_y': cell_info['centroid_y'].values,
        'area': cell_info['area'].values,
        'sample': sample_name,
    })
    obs.index = [f'{sample_name}_{cid}' for cid in obs['cell_id']]
    obs.index.name = None
    
    # --- Build .var ---
    var = pd.DataFrame({
        'marker_name': markers,
        'channel_index': channel_indices,
        'marker_type': [marker_type[m] for m in markers],
    }, index=markers)
    var.index.name = None
    
    # --- Build .obsm['spatial'] ---
    # regionprops centroid returns (row, col) = (vertical, horizontal)
    # centroid_x in CSV = row (vertical), centroid_y = col (horizontal)
    # For spatial convention (x_horizontal, y_vertical): use (centroid_y, centroid_x)
    spatial = np.column_stack([
        cell_info['centroid_y'].values,  # x (horizontal)
        cell_info['centroid_x'].values,  # y (vertical)
    ])
    
    # --- Build AnnData ---
    adata = ad.AnnData(
        X=X,
        obs=obs,
        var=var,
    )
    
    adata.obsm['spatial'] = spatial
    
    # --- Layers ---
    adata.layers['raw'] = X.copy()
    adata.layers['arcsinh'] = np.arcsinh(X / cofactor)
    
    # --- .uns metadata ---
    adata.uns['core_id'] = sample_name
    adata.uns['modality'] = 'IMC'
    adata.uns['pixel_size_um'] = 1.0
    adata.uns['arcsinh_cofactor'] = cofactor
    adata.uns['n_markers'] = len(markers)
    
    # --- Save ---
    out_path = os.path.join(adata_out_dir, f'adata_{sample_name}.h5ad')
    adata.write_h5ad(out_path)
    
    print(f'  {sample_name}: {n_cells} cells, {X.shape[1]} markers -> {out_path}')

print(f'\nDone! Saved {len(roi_filtered)} AnnData files to {adata_out_dir}')

## Verification

Load one file and verify structure, cell counts, and spatial coordinates.

In [ ]:
# Load the largest ROI as verification
test_roi = roi_filtered.iloc[0]['sample_name']
test_path = os.path.join(adata_out_dir, f'adata_{test_roi}.h5ad')

adata_test = ad.read_h5ad(test_path)
print(f'Loaded: {test_path}')
print(f'  Shape: {adata_test.shape}')
print(f'  .obs columns: {list(adata_test.obs.columns)}')
print(f'  .var columns: {list(adata_test.var.columns)}')
print(f'  .obsm keys: {list(adata_test.obsm.keys())}')
print(f'  .layers keys: {list(adata_test.layers.keys())}')
print(f'  .uns keys: {list(adata_test.uns.keys())}')
print(f'  spatial shape: {adata_test.obsm["spatial"].shape}')
print(f'  X dtype: {adata_test.X.dtype}')
print(f'  X range: [{adata_test.X.min():.1f}, {adata_test.X.max():.1f}]')
print(f'  arcsinh range: [{adata_test.layers["arcsinh"].min():.2f}, {adata_test.layers["arcsinh"].max():.2f}]')

# Verify cell count matches CSV
csv_count = pd.read_csv(os.path.join(cell_info_dir, f'{test_roi}_cell_info.csv')).shape[0]
assert adata_test.n_obs == csv_count, f'Cell count mismatch: {adata_test.n_obs} vs {csv_count}'
print(f'  Cell count verified: {adata_test.n_obs} == CSV ({csv_count})')

In [ ]:
import matplotlib.pyplot as plt

# Spatial scatter plot colored by GFAP expression (arcsinh-transformed)
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# GFAP
marker_idx = markers.index('GFAP')
sc = axes[0].scatter(
    adata_test.obsm['spatial'][:, 0],
    adata_test.obsm['spatial'][:, 1],
    c=adata_test.layers['arcsinh'][:, marker_idx],
    s=1, cmap='viridis'
)
axes[0].set_title(f'{test_roi} - GFAP (arcsinh)')
axes[0].set_aspect('equal')
axes[0].invert_yaxis()
plt.colorbar(sc, ax=axes[0])

# CD45
marker_idx = markers.index('CD45')
sc = axes[1].scatter(
    adata_test.obsm['spatial'][:, 0],
    adata_test.obsm['spatial'][:, 1],
    c=adata_test.layers['arcsinh'][:, marker_idx],
    s=1, cmap='viridis'
)
axes[1].set_title(f'{test_roi} - CD45 (arcsinh)')
axes[1].set_aspect('equal')
axes[1].invert_yaxis()
plt.colorbar(sc, ax=axes[1])

# DNA
marker_idx = markers.index('DNA')
sc = axes[2].scatter(
    adata_test.obsm['spatial'][:, 0],
    adata_test.obsm['spatial'][:, 1],
    c=adata_test.layers['arcsinh'][:, marker_idx],
    s=1, cmap='viridis'
)
axes[2].set_title(f'{test_roi} - DNA (arcsinh)')
axes[2].set_aspect('equal')
axes[2].invert_yaxis()
plt.colorbar(sc, ax=axes[2])

plt.tight_layout()
plt.show()

In [ ]:
# Summary table of all generated AnnData files
summary = []
for f in sorted(os.listdir(adata_out_dir)):
    if f.endswith('.h5ad'):
        a = ad.read_h5ad(os.path.join(adata_out_dir, f))
        summary.append({
            'file': f,
            'n_cells': a.n_obs,
            'n_markers': a.n_vars,
            'sample': a.uns['core_id'],
        })

summary_df = pd.DataFrame(summary)
print(f'Total: {len(summary_df)} files, {summary_df["n_cells"].sum()} cells')
summary_df